In [1]:
import pandas as pd

def load_transcription_data(file_path: str) -> pd.DataFrame | None:
    """
    Loads transcription data from a CSV file into a pandas DataFrame.

    Args:
        file_path: The path to the CSV file.

    Returns:
        A pandas DataFrame containing the data, or None if the file is not found.
    """
    try:
        # Read the CSV file into a DataFrame
        df = pd.read_csv(file_path)
        print("File loaded successfully!")
        return df
    except FileNotFoundError:
        print(f"Error: The file was not found at the path: {file_path}")
        print("Please make sure the file is uploaded and the path is correct.")
        return None

# --- Usage Example ---

# Define the name of your file
# This assumes the file is in the same directory as your script
file_name = '/teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/02_intermediate/transcripts/mustafaa4a_ASR-Somali/transcriptions_database.csv'

# Load the data using the function
transcriptions_df = load_transcription_data(file_name)

# If the DataFrame was loaded successfully, display the first 5 rows
if transcriptions_df is not None:
    print("\nHere's a preview of your data:")
    print(transcriptions_df.head())

File loaded successfully!

Here's a preview of your data:
                                 id  \
0  be400344882bded3b69613c55f924523   
1  8919804ce6600b7af5526b91eb18406c   
2  0e8501bdbeb8819eeafac6ddc75efa59   
3  ebf1d24a75fa8d90a2bf4605ec33f62a   
4  192b2896ecdc6563ca66d6671e50c16c   

                                                 url                  title  \
0  https://soundcloud.com/radio-ergo/idaacadda-01...  IDAACADDA 01-JAN-2020   
1  https://soundcloud.com/radio-ergo/idaacadda-03...  IDAACADDA 03-JAN-2020   
2  https://soundcloud.com/radio-ergo/idaacadda-04...  IDAACADDA 04-JAN-2020   
3  https://soundcloud.com/radio-ergo/idaacadda-05...  IDAACADDA 05-JAN-2020   
4  https://soundcloud.com/radio-ergo/idaacadda-06...  IDAACADDA 06-JAN-2021   

   date_recorded              date_processed  processing_duration_seconds  \
0       20200102  2025-10-08T10:22:06.380552                    44.178506   
1       20200103  2025-10-08T10:24:19.056455                    41.640142   
2

In [2]:
transcriptions_df

,id,url,title,date_recorded,date_processed,processing_duration_seconds,audio_size_mb,audio_duration_seconds,transcript_length_chars,transcript_length_words,transcript_text
0,be400344882bded3b69613c55f924523,https://soundcloud.com/radio-ergo/idaacadda-01...,IDAACADDA 01-JAN-2020,20200102,2025-10-08T10:22:06.380552,44.178506,54.933331,3600.049,44796,6371,halkani waa raadyahay ergo ee codka arrimahaab...
1,8919804ce6600b7af5526b91eb18406c,https://soundcloud.com/radio-ergo/idaacadda-03...,IDAACADDA 03-JAN-2020,20200103,2025-10-08T10:24:19.056455,41.640142,54.926007,3599.569,44992,6467,halkani waa raadyahay ergo ee codka arrimahaab...
2,0e8501bdbeb8819eeafac6ddc75efa59,https://soundcloud.com/radio-ergo/idaacadda-04...,IDAACADDA 04-JAN-2020,20200104,2025-10-08T10:26:33.998488,41.677658,54.921247,3599.257,50898,7505,halkani waa raadyahay ergo ee codka arrimahaab...
3,ebf1d24a75fa8d90a2bf4605ec33f62a,https://soundcloud.com/radio-ergo/idaacadda-05...,IDAACADDA 05-JAN-2020,20200105,2025-10-08T10:28:43.726456,41.240980,54.701154,3584.833,46206,6553,halkani waa rahadyaha ergo ee codka arrimaha b...
4,192b2896ecdc6563ca66d6671e50c16c,https://soundcloud.com/radio-ergo/idaacadda-06...,IDAACADDA 06-JAN-2021,20210110,2025-10-08T10:30:58.274662,41.394951,54.869245,3595.849,50599,7569,halkani waa raadyahay ergo ee codka arrimahaab...
...,...,...,...,...,...,...,...,...,...,...,...
1682,4457454822918e92ca861774b64637cb,https://soundcloud.com/radio-ergo/idaacadda-25...,IDAACADDA 25-SEP-2025,20250925,2025-10-10T22:18:19.848479,26.755927,54.854230,3594.867,54992,8029,halkani waa raadyahay rgoee codka arrimaha ban...
1683,03b134754e2f2e3d7f76c5fd51c7f696,https://soundcloud.com/radio-ergo/idaacadda-26...,IDAACADDA 26-SEP-2025,20250926,2025-10-10T22:20:23.250778,25.964677,54.032453,3541.029,51210,7582,halkani waa raadyahay rgoee codka arrimaha bni...
1684,0a71a07ed44837e6434185d4caccfcef,https://soundcloud.com/radio-ergo/idaacadda-28...,IDAACADDA 28-SEP-2025,20250928,2025-10-10T22:22:27.932304,25.535577,54.386212,3564.199,49310,6936,halkani waa raadyahy rgoee codka arrimaha bani...
1685,76950408579c0358825e1daa904f3d35,https://soundcloud.com/radio-ergo/idaacadda-29...,IDAACADDA 29-SEP-2025,20250929,2025-10-10T22:24:30.973702,26.277499,54.391339,3564.539,57220,8280,halkani waa raadyahay rgoee codka arrimaha ban...


In [3]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from typing import List, Tuple
import warnings
import re
warnings.filterwarnings('ignore')

def load_nllb_model(model_name: str = "facebook/nllb-200-3.3B") -> tuple:
    """
    Load NLLB-200 translation model and tokenizer with optimization.
    """
    print(f"Loading {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang="som_Latn")
    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
    )
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    model.eval()
    print(f"Model loaded on device: {device}")
    
    return tokenizer, model, device

def split_into_sentences(text: str) -> List[str]:
    """
    Improved sentence splitting for Somali text.
    Falls back to word-based chunking if sentence detection fails.
    """
    # Try multiple splitting patterns
    # Pattern 1: Space after common Somali words that end sentences
    sentences = re.split(r'\s+(?=waxaa|waxa|marka|haddii|sida|taasi)', text, flags=re.IGNORECASE)
    
    # If we got good splits, return them
    if len(sentences) > 5:
        return [s.strip() for s in sentences if s.strip()]
    
    # Pattern 2: Simple period/space splits
    sentences = re.split(r'\.\s+', text)
    if len(sentences) > 3:
        return [s.strip() for s in sentences if s.strip()]
    
    # Pattern 3: Split by common conjunctions
    sentences = re.split(r'\s+(ayaa|oo|iyo|ee)\s+', text, flags=re.IGNORECASE)
    
    # Recombine with conjunctions
    result = []
    for i in range(0, len(sentences), 2):
        if i+1 < len(sentences):
            result.append(sentences[i] + ' ' + sentences[i+1])
        else:
            result.append(sentences[i])
    
    if len(result) > 2:
        return [s.strip() for s in result if s.strip()]
    
    # Fallback: Split by word count (every ~50 words)
    words = text.split()
    chunk_size = 50
    sentences = []
    for i in range(0, len(words), chunk_size):
        chunk = ' '.join(words[i:i+chunk_size])
        sentences.append(chunk)
    
    return [s.strip() for s in sentences if s.strip()]

def create_semantic_chunks(
    text: str,
    tokenizer: AutoTokenizer,
    max_tokens: int = 450,
    overlap_sentences: int = 2
) -> List[Tuple[str, int, int]]:
    """
    Split text into chunks that respect 512 token limit.
    Uses aggressive splitting if needed.
    """
    sentences = split_into_sentences(text)
    
    if not sentences:
        return [(text, 0, len(text))]
    
    print(f"  -> Found {len(sentences)} sentences/segments")
    
    chunks = []
    current_chunk = []
    current_tokens = 0
    
    i = 0
    while i < len(sentences):
        sentence = sentences[i]
        
        # Count tokens for this sentence
        sentence_tokens = len(tokenizer.encode(sentence, add_special_tokens=True))
        
        # If single sentence exceeds max, split by words
        if sentence_tokens > max_tokens:
            print(f"  -> Sentence {i+1} has {sentence_tokens} tokens, splitting by words...")
            words = sentence.split()
            word_chunk = []
            word_tokens = 0
            
            for word in words:
                word_token_count = len(tokenizer.encode(word + ' ', add_special_tokens=False))
                
                if word_tokens + word_token_count > max_tokens - 10:  # Leave buffer
                    if word_chunk:
                        chunk_text = ' '.join(word_chunk)
                        chunks.append((chunk_text, 0, 0))
                        print(f"     -> Created word chunk: {len(tokenizer.encode(chunk_text, add_special_tokens=True))} tokens")
                    word_chunk = [word]
                    word_tokens = word_token_count
                else:
                    word_chunk.append(word)
                    word_tokens += word_token_count
            
            if word_chunk:
                chunk_text = ' '.join(word_chunk)
                chunks.append((chunk_text, 0, 0))
                print(f"     -> Created final word chunk: {len(tokenizer.encode(chunk_text, add_special_tokens=True))} tokens")
            i += 1
            continue
        
        # Check if adding this sentence exceeds limit
        if current_tokens + sentence_tokens > max_tokens:
            # Save current chunk
            if current_chunk:
                chunk_text = ' '.join(current_chunk)
                chunks.append((chunk_text, 0, 0))
                print(f"  -> Created chunk {len(chunks)}: {len(tokenizer.encode(chunk_text, add_special_tokens=True))} tokens")
            
            # Start new chunk with overlap
            overlap_start = max(0, len(current_chunk) - overlap_sentences)
            current_chunk = current_chunk[overlap_start:]
            current_tokens = sum(
                len(tokenizer.encode(s, add_special_tokens=True)) 
                for s in current_chunk
            )
        
        current_chunk.append(sentence)
        current_tokens += sentence_tokens
        i += 1
    
    # Add final chunk
    if current_chunk:
        chunk_text = ' '.join(current_chunk)
        chunks.append((chunk_text, 0, 0))
        print(f"  -> Created final chunk {len(chunks)}: {len(tokenizer.encode(chunk_text, add_special_tokens=True))} tokens")
    
    return chunks

def deduplicate_overlap(chunks: List[str], overlap_threshold: int = 5) -> str:
    """
    Intelligently merge chunks by detecting and removing overlapping content.
    """
    if not chunks:
        return ""
    
    if len(chunks) == 1:
        return chunks[0]
    
    result = [chunks[0]]
    
    for i in range(1, len(chunks)):
        prev_chunk = result[-1]
        curr_chunk = chunks[i]
        
        prev_words = prev_chunk.split()
        curr_words = curr_chunk.split()
        
        # Find overlap
        max_overlap = min(len(prev_words), len(curr_words), 15)
        overlap_found = 0
        
        for overlap_size in range(max_overlap, overlap_threshold - 1, -1):
            prev_end = ' '.join(prev_words[-overlap_size:])
            curr_start = ' '.join(curr_words[:overlap_size])
            
            if prev_end.lower() == curr_start.lower():
                overlap_found = overlap_size
                break
        
        # Merge with overlap removed
        if overlap_found > 0:
            result.append(' '.join(curr_words[overlap_found:]))
        else:
            result.append(curr_chunk)
    
    return ' '.join(result)

def translate_text_chunked(
    text: str,
    tokenizer: AutoTokenizer,
    model: AutoModelForSeq2SeqLM,
    device: str,
    src_lang: str = "som_Latn",
    tgt_lang: str = "eng_Latn",
    max_tokens: int = 450,
    overlap_sentences: int = 2
) -> str:
    """
    Translate long text with proper chunking for 512 token architecture.
    """
    if not text or not isinstance(text, str):
        raise ValueError("Text must be a non-empty string")
    
    # Split into semantic chunks
    chunks = create_semantic_chunks(
        text=text,
        tokenizer=tokenizer,
        max_tokens=max_tokens,
        overlap_sentences=overlap_sentences
    )
    
    print(f"  -> Split into {len(chunks)} chunks for translation")
    
    # Translate each chunk
    translated_chunks = []
    tokenizer.src_lang = src_lang
    tgt_lang_id = tokenizer.convert_tokens_to_ids(tgt_lang)
    
    for idx, (chunk_text, _, _) in enumerate(chunks):
        inputs = tokenizer(
            chunk_text,
            return_tensors="pt",
            max_length=512,
            truncation=True,
            padding=True
        ).to(device)
        
        with torch.no_grad():
            translated_tokens = model.generate(
                **inputs,
                forced_bos_token_id=tgt_lang_id,
                max_length=512,
                num_beams=5,
                length_penalty=1.0,
                early_stopping=True,
                no_repeat_ngram_size=3,
                repetition_penalty=1.3,
                do_sample=False
            )
        
        translation = tokenizer.batch_decode(
            translated_tokens,
            skip_special_tokens=True
        )[0]
        
        translated_chunks.append(translation)
        print(f"  -> Translated chunk {idx + 1}/{len(chunks)} ({len(translation)} chars)")
    
    # Merge chunks with deduplication
    full_translation = deduplicate_overlap(translated_chunks)
    
    return full_translation

def batch_translate_dataframe(
    df: pd.DataFrame,
    text_column: str,
    tokenizer: AutoTokenizer,
    model: AutoModelForSeq2SeqLM,
    device: str,
    new_column_name: str = "transcript_text_english",
    src_lang: str = "som_Latn",
    tgt_lang: str = "eng_Latn",
    use_chunking: bool = True
) -> pd.DataFrame:
    """
    Translate a text column in DataFrame.
    """
    if text_column not in df.columns:
        raise KeyError(f"Column '{text_column}' not found in DataFrame")
    
    df_translated = df.copy()
    translations = []
    
    print(f"Translating {len(df)} rows...")
    for idx, text in enumerate(df[text_column]):
        try:
            print(f"\nRow {idx + 1}/{len(df)}")
            
            if use_chunking:
                translation = translate_text_chunked(
                    text=text,
                    tokenizer=tokenizer,
                    model=model,
                    device=device,
                    src_lang=src_lang,
                    tgt_lang=tgt_lang
                )
            else:
                tokenizer.src_lang = src_lang
                inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True).to(device)
                tgt_lang_id = tokenizer.convert_tokens_to_ids(tgt_lang)
                
                with torch.no_grad():
                    translated_tokens = model.generate(
                        **inputs,
                        forced_bos_token_id=tgt_lang_id,
                        max_length=512,
                        num_beams=5,
                        no_repeat_ngram_size=3,
                        repetition_penalty=1.3
                    )
                
                translation = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
            
            translations.append(translation)
            print(f"✓ Completed row {idx + 1}")
                
        except Exception as e:
            print(f"✗ Error translating row {idx}: {str(e)}")
            translations.append(None)
    
    df_translated[new_column_name] = translations
    return df_translated


# Usage example
print("Starting optimized translation pipeline (512 token limit)...\n")

# Load model
tokenizer, model, device = load_nllb_model("facebook/nllb-200-3.3B")

# Analyze chunking
print("\n" + "="*80)
print("CHUNKING ANALYSIS - 512 Token Architecture")
print("="*80)

sample_text = transcriptions_df.iloc[0]['transcript_text']
print(f"Sample text: {len(sample_text)} chars, {len(sample_text.split())} words\n")

chunks = create_semantic_chunks(sample_text, tokenizer, max_tokens=450, overlap_sentences=2)
print(f"\nFinal chunk count: {len(chunks)}")
print(f"Average chunk size: {sum(len(c[0]) for c in chunks) / len(chunks):.0f} chars")

# Show first 3 chunks
print(f"\nFirst 3 chunk details:")
for i, (chunk, _, _) in enumerate(chunks[:3]):
    token_count = len(tokenizer.encode(chunk, add_special_tokens=True))
    print(f"  Chunk {i+1}: {token_count} tokens, {len(chunk)} chars")
    print(f"    First 80 chars: {chunk[:80]}...")

print("="*80)

# Test translation
test_df = transcriptions_df.head(1).copy()

print(f"\nTranslating {len(test_df)} test row...")
test_df_translated = batch_translate_dataframe(
    df=test_df,
    text_column="transcript_text",
    tokenizer=tokenizer,
    model=model,
    device=device,
    new_column_name="transcript_text_english",
    use_chunking=True
)

# Display results
print("\n" + "="*80)
print("TRANSLATION RESULTS")
print("="*80)

for idx, row in test_df_translated.iterrows():
    original = row['transcript_text']
    translated = row['transcript_text_english'] if row['transcript_text_english'] else ""
    
    print(f"\n--- Row {idx} ---")
    print(f"Title: {row['title']}")
    print(f"Original: {len(original)} chars, {len(original.split())} words")
    print(f"Translated: {len(translated)} chars, {len(translated.split())} words")
    print(f"Compression ratio: {len(translated)/len(original):.1%}")
    
    words = translated.split()
    unique_words = len(set(words))
    diversity = unique_words / len(words) if words else 0
    print(f"Vocabulary diversity: {diversity:.1%} ({unique_words} unique / {len(words)} total)")
    
    print(f"\nFirst 500 chars (English):\n{translated[:500]}...")
    print(f"\nLast 300 chars (English):\n...{translated[-300:]}")

print("\n" + "="*80)
avg_ratio = test_df_translated['transcript_text_english'].str.len().mean() / test_df_translated['transcript_text'].str.len().mean()
print(f"Translation ratio: {avg_ratio:.1%} ({'✓ Good' if 0.45 <= avg_ratio <= 0.75 else '⚠ Review' if avg_ratio > 0 else '✗ Failed'})")
print("="*80)

Starting optimized translation pipeline (512 token limit)...

Loading facebook/nllb-200-3.3B...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded on device: cuda

CHUNKING ANALYSIS - 512 Token Architecture
Sample text: 44796 chars, 6371 words

  -> Found 204 sentences/segments
  -> Created chunk 1: 403 tokens
  -> Created chunk 2: 399 tokens
  -> Created chunk 3: 356 tokens
  -> Created chunk 4: 438 tokens
  -> Created chunk 5: 415 tokens
  -> Created chunk 6: 324 tokens
  -> Created chunk 7: 422 tokens
  -> Created chunk 8: 429 tokens
  -> Created chunk 9: 412 tokens
  -> Created chunk 10: 379 tokens
  -> Created chunk 11: 419 tokens
  -> Created chunk 12: 430 tokens
  -> Created chunk 13: 286 tokens
  -> Created chunk 14: 400 tokens
  -> Created chunk 15: 360 tokens
  -> Created chunk 16: 365 tokens
  -> Created chunk 17: 285 tokens
  -> Created chunk 18: 371 tokens
  -> Created chunk 19: 335 tokens
  -> Created chunk 20: 408 tokens
  -> Created chunk 21: 431 tokens
  -> Created chunk 22: 406 tokens
  -> Created chunk 23: 378 tokens
  -> Created chunk 24: 404 tokens
  -> Created chunk 25: 432 tokens
  -> Created c

In [4]:
test_df_translated

,id,url,title,date_recorded,date_processed,processing_duration_seconds,audio_size_mb,audio_duration_seconds,transcript_length_chars,transcript_length_words,transcript_text,transcript_text_english
0,be400344882bded3b69613c55f924523,https://soundcloud.com/radio-ergo/idaacadda-01...,IDAACADDA 01-JAN-2020,20200102,2025-10-08T10:22:06.380552,44.178506,54.933331,3600.049,44796,6371,halkani waa raadyahay ergo ee codka arrimahaab...,This is the tracker of the voice of humanitari...


In [5]:
full_transcript_text = test_df_translated.loc[0, 'transcript_text']

In [6]:
full_transcript_text

"halkani waa raadyahay ergo ee codka arrimahaabiniaadamnimada uu fodhigiisu yahay magaaladana irobia ee dalka kenya waxad naga dageysanaysaan mujadda gaaban ee dharrkeedu yahay kooy labaatanka mitrbanuna dhegenta toddob iyo toban kunsideed boqolshon iyo fartan magaartes sacadda geeska afrikada barimarka ay tahay saddexda ilaa afarta galbnimo waxaad sidoo kale naga dhagaysanaysaan qaar ka tirsan idaacdaha dalka iyo barta aynuu ku leenahay interneta hee fadhigiisu yahay rdh iyo ergo dhoodta waarc edu daxdaer aadaadeddowwkulantigucana dhagastiyal manto arbacoah wax kowda bishacanaayo sanadka labada kun iyo labaatanka sanad cusubna dhammaantiin meelkastood joogta kunaaso dhowwada idaacoddeni o qodobada ah maanta an idinku hayno ay ka mid yihiin warbixin xog uruurin ah oo puntland ay soo saartay ayay ku sheegaysaa in dhimashada ah hooyada uur kale ee xilliga dhalmada ay hoos u dhacday marka loo eego halka ay taagnayd shansano ka hor sida kale waxaan idiina haynaa intixaano lagu ogaanayo tay

In [7]:
full_transcript_text_english = test_df_translated.loc[0, 'transcript_text_english']

In [8]:
full_transcript_text_english

'This is the tracker of the voice of humanitarian causes headquartered in Nairobi, Kenya. You\'ll hear from us on the short-sleeved, 20-foot-tall, 17,800-foot-long radio station in Cape Town. Primary is from 3:00 to 4:00 p.m. And you\'ll also hear us on some of the country\'s radio stations and on our website, which is hosted by RTH, and the message of enduring humanity. On the first Wednesday of August, we celebrate the new moon of the year two thousand and twenty-five. We also have a test to determine the quality of teachers in the western and central regions of Somalia. Braamijhkanatacap is currently underway. The livestock bill as well as voices from the audience will hear this. I\'m Mohamed Hassan. I developed the program for you, but let\'s start with a news story today that will be read by Pasheer Pal. Remind the media of the need to address this unprecedented number of women and girls who are facing a major crisis. Now the humanitarian agencies and civil society organizations w

Analysis of the Provided Translation:


In [ ]:
# # Translate full dataset (will take several hours on GPU)
# full_df_translated = batch_translate_dataframe(
#     df=transcriptions_df,
#     text_column="transcript_text",
#     tokenizer=tokenizer,
#     model=model,
#     device=device,
#     use_chunking=True
# )

# # Save progress periodically
# full_df_translated.to_csv('transcriptions_translated.csv', index=False)